# Paper III: verified Taiwan dataset and reviewer experiment smoke tests
This notebook checks the original dataset checksum, then runs one seed of each controlled experiment. It does not establish full-study results.

In [ ]:
import os, hashlib, subprocess, sys, urllib.request
from pathlib import Path
root=Path('/content/paper3_review')
root.mkdir(exist_ok=True)
base='https://raw.githubusercontent.com'
dataset_url=base+'/neurospin/pystatsml/master/datasets/default%20of%20credit%20card%20clients.xls'
script_url=base+'/Imomazin/research-projects/main/paper3/paper3_controlled_experiments_corrected.py'
data=root/'default of credit card clients.xls'
script=root/'paper3_controlled_experiments_corrected.py'
for url,path in [(dataset_url,data),(script_url,script)]:
    urllib.request.urlretrieve(url,path)
    print(path.name,path.stat().st_size)
expected='30c6be3abd8dcfd3e6096c828bad8c2f011238620f5369220bd60cfc82700933'
actual=hashlib.sha256(data.read_bytes()).hexdigest()
print('SHA256',actual)
assert actual==expected,'DATASET HASH MISMATCH: STOP'
subprocess.check_call([sys.executable,'-m','pip','-q','install','xlrd'])
import pandas as pd
df=pd.read_excel(data,sheet_name='Data',header=1)
assert df.shape==(30000,25)
assert df['default payment next month'].sum()==6636
print('VERIFIED ORIGINAL DATASET',df.shape)
for mode in ['matched','control','calibration']:
    out=root/(mode+'_seed1.csv')
    cmd=[sys.executable,str(script),'--mode',mode,'--data',str(data),'--seeds','1','--out',str(out)]
    print('RUN',mode,flush=True)
    subprocess.check_call(cmd)
    print('SAVED',out,'rows',len(pd.read_csv(out)),flush=True)
print('SMOKE TESTS COMPLETE. Do not interpret as 100-seed findings.')
